# C Monte Carlo Simulation

This notebook compiles a small C kernel for geometric Brownian motion simulation, calls it from Python, compares the output distribution against the Python/NumPy simulator, and runs a simple benchmark.

## Why C is reasonable here

Monte Carlo simulation is a repeated numerical loop. For each path and each time step, the code generates a random shock, computes a log return, and updates the stock price. That kind of repeated loop is a natural place to try C.

This section is optional. NumPy is already very fast because much of NumPy is compiled code underneath. The C kernel is mainly useful here for learning, portfolio differentiation, and showing that numerical kernels can be separated from the Python workflow.

## Random number generation

The C file uses a simple xorshift64* generator and the Box-Muller transform to create normal random variables. This is fine for a learning project, but it should not be presented as a production-grade risk RNG.

Reproducibility matters because the same seed should generate the same C paths every time. Python and C will not match path-by-path because NumPy uses a different RNG.

In [ ]:
from pathlib import Path
import platform
import shutil
import subprocess
import sys

import pandas as pd

In [ ]:
project_root = Path.cwd()

if not (project_root / "c_src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

## Compile the C library

In [ ]:
compiled_dir = project_root / "compiled"
compiled_dir.mkdir(exist_ok=True)

compiler = shutil.which("gcc") or shutil.which("clang")

if compiler is None:
    raise RuntimeError("No C compiler found. Install gcc or clang before running this notebook.")

system_name = platform.system().lower()

if system_name == "darwin":
    library_path = compiled_dir / "libmonte_carlo.dylib"
    compile_command = [
        compiler,
        "-O3",
        "-fPIC",
        "-dynamiclib",
        str(project_root / "c_src" / "monte_carlo.c"),
        "-o",
        str(library_path),
    ]
elif system_name == "windows":
    library_path = compiled_dir / "monte_carlo.dll"
    compile_command = [
        compiler,
        "-O3",
        "-shared",
        str(project_root / "c_src" / "monte_carlo.c"),
        "-o",
        str(library_path),
    ]
else:
    library_path = compiled_dir / "libmonte_carlo.so"
    compile_command = [
        compiler,
        "-O3",
        "-fPIC",
        "-shared",
        str(project_root / "c_src" / "monte_carlo.c"),
        "-o",
        str(library_path),
        "-lm",
    ]

subprocess.run(compile_command, check=True)
library_path

## Run Python and C simulations

In [ ]:
from src.c_bindings import (
    MonteCarloCLibrary,
    benchmark_python_vs_c,
    compare_python_and_c_statistics,
)
from src.simulation import simulate_gbm_paths

starting_price = 100.0
drift = 0.06
volatility = 0.25
time_horizon = 30 / 365.25
steps = 30
num_paths = 10_000
random_seed = 42

mc_c = MonteCarloCLibrary(library_path)

python_paths = simulate_gbm_paths(
    starting_price=starting_price,
    drift=drift,
    volatility=volatility,
    time_horizon=time_horizon,
    steps=steps,
    num_paths=num_paths,
    random_seed=random_seed,
)

c_result = mc_c.simulate_gbm_paths(
    starting_price=starting_price,
    drift=drift,
    volatility=volatility,
    time_horizon=time_horizon,
    steps=steps,
    num_paths=num_paths,
    random_seed=random_seed,
)

c_paths = c_result.paths

c_result.status, c_paths.head()

## Compare simulation statistics

In [ ]:
comparison = compare_python_and_c_statistics(
    python_paths=python_paths,
    c_paths=c_paths,
)

comparison

In [ ]:
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

c_paths.iloc[:, :12].to_csv(processed_dir / "c_generated_simulated_paths_sample.csv", index=False)
comparison.to_csv(processed_dir / "python_vs_c_simulation_statistics.csv", index=False)

## Benchmark

A fair benchmark excludes compilation time, uses the same model inputs, and makes both implementations write full path arrays. It should not compare a tiny scalar C call against a vectorized NumPy operation and then overclaim the result.

In [ ]:
benchmark = benchmark_python_vs_c(
    c_library=mc_c,
    starting_price=starting_price,
    drift=drift,
    volatility=volatility,
    time_horizon=time_horizon,
    steps=252,
    num_paths=50_000,
    random_seed=random_seed,
    number=3,
)

benchmark

In [ ]:
benchmark.to_csv(processed_dir / "python_vs_c_monte_carlo_benchmark.csv", index=False)

## Interpretation

If C is faster in your benchmark, report the measured result with the exact parameters. If NumPy is similar or faster, that is still a good result because it shows honest benchmarking. The point is not to force a speed claim. The point is to understand where C can help and how to validate it.